## Objectives

- Explain MEV through transactions in the wild
- Performing an atomic arbitrage transaction across different pools of Uniswap

## Prerequisites

- Metamask extension should be installed on the browser
- An account with a minimum of 0.1 SepoliaETH, 10 USTUSD. 

## GUI interaction & MEV Overview

**Maximal Extractable Value (MEV)** refers to the maximum value that can be extracted from block production in excess of the standard block reward and gas fees. This is achieved by including, excluding, and changing the order of transactions in a block.

* Go to MEV explorer [here](https://eigenphi.io/) to visualize real-time on-chain activities.
* Mostly covers arbitrage, sandwich attacks, and JIT liquidity attacks. These visualization tools allow users and researchers to track how "Searchers" (automated bots) extract value from the public mempool.

The following are some detailed examples of MEV transactions.

### Arbitrage transaction

Exploiting price differences across different markets to make a riskless profit – this is usually benign and highly beneficial for market efficiency.

* **Mechanism:** Search two or more Decentralized Exchanges (DEXs like Uniswap and SushiSwap) for temporary differences in asset prices.
* **Execution:** Trade the asset between them to gain tokens in the numeraire (the base asset, such as ETH or USDC). Since these trades are often executed atomically via smart contracts, the risk of partial execution is entirely eliminated.
* Consider an example of an arbitrage transaction [here](https://etherscan.io/tx/0xb6ae673a7d27e89c263afce8ef20966b4e5684c08671a99f1d44ba440aec8f73).

**Example:**

<img src="figs/fig1.png " alt="arbitrage transaction" width="80%"/>

296.92 WETH -> 31 million FODL; 31 million FODL -> 3.27 million USDC; 3.27 million USDC -> 837.26 WETH;

Final profit = 837.26 - 296.92 = 540.34 WETH

### Liquidation transaction

Like "margin calls" in traditional finance. Because DeFi lending protocols cannot rely on traditional credit scores, they require over-collateralization. Here, the miner or searcher spots lending positions that are underwater (where the collateral value drops dangerously close to the borrowed debt value) and liquidates them to earn a commission.

* **Process:** Liquidation involves notifying the lending protocol (e.g., Aave, Compound) that the value of the "collateral" posted by the borrower has fallen below the required health threshold.
* **Outcome:** This makes the protocol take possession of the collateral, sell it off to repay the debt, and give a handsome commission (liquidation bounty) to the liquidator as a reward for securing the protocol.
* An example of a liquidation is shown [here](https://etherscan.io/tx/0xd70b42daec5bb9ac6e5df3d25d309f186db50df701f667e1f20b22448ea27d41).


**Example:**

<img src="figs/fig2.png " alt="liquidation transaction" width="80%"/>

**Step 1: Trigger Flash Loan (Borrow WBTC)**
* **Action:** The liquidator (address `0x3333...`) utilizes SushiSwap's Flash Swap feature to borrow 385.36 WBTC with zero upfront collateral.

**Step 2: Repay Debt (WBTC → cWBTC)**
* **Action:** The liquidator sends the borrowed WBTC to the Compound protocol.
* **Purpose:** This is executed to repay the debt on behalf of a borrower whose collateralization ratio has dropped below the required safety threshold (making the position eligible for liquidation).

**Step 3: Seize Collateral (Receive cETH)**
* **Action:** By repaying the underwater debt, the liquidator successfully seizes the liquidated user's (address `0xdf63...`) collateral at a protocol-defined discount, receiving approximately 715,000 cETH (Compound's interest-bearing ETH token).

**Step 4: Redeem Underlying Assets (cETH → WETH/ETH)**
* **Action:** The liquidator interacts with Compound a second time, burning the 715,000 cETH tokens to redeem the underlying base assets (ETH).

**Step 5: Repay Flash Loan (Pay WETH)**
* **Action:** The liquidator converts the redeemed ETH into WETH and transfers 12,841.22 WETH back to SushiSwap.



### Sandwich transaction

Sandwiching is the act of introducing transactions before and after a user's swap to earn a risk-free profit by exploiting the user's "slippage tolerance" (the maximum price change a user is willing to accept).

* **The Setup:** Attackers monitor the public mempool for pending large trades that will significantly move the price of an asset.
* **Frontrunning (Insertion before):** This is bad for the user and malicious. The attacker bribes the block builder to place their buy order *before* the victim's trade, buying the asset for cheap.
* **Victim execution:** Let the victim's transaction incur, which pushes the asset's price artificially high.
* **Backrunning (Insertion after):** Sell back the original asset immediately after the victim's trade at the newly inflated price. This part is considered benign since it does not give anyone a worse price and stabilizes the market back to its true peg.
* Consider the transactions for a sandwich attack [here](https://etherscan.io/txs?block=15449712&p=6) - as described in [this article](https://medium.com/@alexanderzaidelson/dissecting-mev-arbitrage-fb8f9492cdf1).

**Example:**

<img src="https://miro.medium.com/v2/resize:fit:1100/format:webp/1*jBLHMliIvBziJhpmfyNDwQ.png" alt="sandwich attack" width="80%"/>

* **Step 1: Mempool Monitoring & Calculation**
An MEV bot detects a pending victim transaction in the mempool (swapping 65.91 ETH for 50,000 BONE). The bot calculates that this large order will significantly pump the price of BONE and notes that the victim has set a high slippage tolerance.

* **Step 2: Front-running (The Top Bun)**
The bot submits a transaction with a high bribe to the block builder, ensuring it executes right before the victim. The bot buys 12,566 BONE for 16.36 ETH, draining pool liquidity and artificially inflating the price of BONE.

* **Step 3: The Victim Trade (The Meat)**
The victim's transaction executes next. Because the bot already pumped the price, the victim's 65.91 ETH buys BONE at a much worse rate. The victim absorbs the slippage loss and pushes the BONE price even higher.

* **Step 4: Back-running (The Bottom Bun)**
The bot's second transaction executes immediately after. With the BONE price now at its peak, the bot instantly dumps the 12,566 BONE it just bought, receiving 16.59 ETH in return.

### JIT (Just-In-Time) liquidity transaction

A highly sophisticated strategy often seen in concentrated liquidity AMMs (like Uniswap V3), where bots "snipe" trading fees from passive Liquidity Providers (LPs).

* **Detection:** Search the mempool for a high-value pending transaction that will generate significant trading fees.
* **Injection:** Provide a massive amount of concentrated liquidity to the pool right before that transaction occurs (often executing within the exact same block).
* **Extraction:** Earn the majority of the swap fees because the pool dynamically allocates fees based on the proportion of liquidity provided at that specific price tick.
* **Withdrawal:** Withdraw the liquidity and the accrued fees immediately post-trade, leaving passive LPs with almost zero fee revenue from that massive trading volume.
* Consider the transactions for a JIT liquidity transactions here: [trans1](https://etherscan.io/tx/0x5c261f27e262c9b06b7b1ac9f2c646c50488756484adf4f38ce9164bfcd71129) and [trans2](https://etherscan.io/tx/0x49cff070709d38fa3e648cf7da3f97c875f6c2dc93226fc7e944fbfc8d409a8f).

**Example:**

<img src="figs/fig3.png" alt="JIT liquidity transaction" width="80%"/>




# Atomic swap for arbitrage across pools (Uniswap v3)

As we learned in week 4, we can perform the atomic swap across different pools of Uniswap. If there is a price difference between the two pools, we can make a profit by swapping in one direction on one pool and then swapping back on the other pool. 

Consider a scenario where an asset is priced lower on one pool of Uniswap than another. An arbitrageur will buy that asset on one of the pools and sell it on the other. However, performing this trade in two steps might open the arbitrageur to a fundamental risk - what if one of the pools no longer accepts the trade at a higher price? In this case, the trade might return a loss. Performing the swaps under an atomic transaction with control logic can remove this risk. In this part of the lab, we will use a contract to perform arbitrage on Uniswap pools atomically.

We will use our ERC20 Lab Tokens, USTUSD, and USTETH across the two exchanges.

With these, we will swap USTUSD->USTETH and then USTETH->USTUSD on Uniswap v3 within one transaction. We will use an advanced version of this technique in future labs to leverage arbitrage at minimal capital requirements.

## Which pools to arbitrage?

We first see the list of pools that exchange USTUSD and USTETH. Then, we work out the best way to do arbitrage so that we make a profit at the end. The following cell should print the address of the two pools we are interested in.

In [ ]:
from web3 import Web3

infura_key = ''
public_address = ''
private_key = ''

w3 = Web3(Web3.HTTPProvider(f'https://sepolia.infura.io/v3/{infura_key}'))

print(w3.eth.get_block_number())
print(w3.__dict__)

In [ ]:
USTUSD_address = Web3.to_checksum_address('0xc83B0efA5B3F13851DfA11de72EF6AFeF026730c')
USTETH_address = Web3.to_checksum_address('0x4E22e9951770b98d4f3B2BA6647f0f40A15A4057')

factory_addr = '0x0227628f3F023bb0B980b67D528571c95c6DaC1c'
abi_factory = '[{"inputs":[],"stateMutability":"nonpayable","type":"constructor"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"uint24","name":"fee","type":"uint24"},{"indexed":true,"internalType":"int24","name":"tickSpacing","type":"int24"}],"name":"FeeAmountEnabled","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"oldOwner","type":"address"},{"indexed":true,"internalType":"address","name":"newOwner","type":"address"}],"name":"OwnerChanged","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"token0","type":"address"},{"indexed":true,"internalType":"address","name":"token1","type":"address"},{"indexed":true,"internalType":"uint24","name":"fee","type":"uint24"},{"indexed":false,"internalType":"int24","name":"tickSpacing","type":"int24"},{"indexed":false,"internalType":"address","name":"pool","type":"address"}],"name":"PoolCreated","type":"event"},{"inputs":[{"internalType":"address","name":"tokenA","type":"address"},{"internalType":"address","name":"tokenB","type":"address"},{"internalType":"uint24","name":"fee","type":"uint24"}],"name":"createPool","outputs":[{"internalType":"address","name":"pool","type":"address"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"uint24","name":"fee","type":"uint24"},{"internalType":"int24","name":"tickSpacing","type":"int24"}],"name":"enableFeeAmount","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"uint24","name":"","type":"uint24"}],"name":"feeAmountTickSpacing","outputs":[{"internalType":"int24","name":"","type":"int24"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"","type":"address"},{"internalType":"address","name":"","type":"address"},{"internalType":"uint24","name":"","type":"uint24"}],"name":"getPool","outputs":[{"internalType":"address","name":"","type":"address"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"owner","outputs":[{"internalType":"address","name":"","type":"address"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"parameters","outputs":[{"internalType":"address","name":"factory","type":"address"},{"internalType":"address","name":"token0","type":"address"},{"internalType":"address","name":"token1","type":"address"},{"internalType":"uint24","name":"fee","type":"uint24"},{"internalType":"int24","name":"tickSpacing","type":"int24"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"_owner","type":"address"}],"name":"setOwner","outputs":[],"stateMutability":"nonpayable","type":"function"}]'

factory_contract = w3.eth.contract(factory_addr, abi=abi_factory)


abi_USTUSD = '[{"inputs":[],"stateMutability":"nonpayable","type":"constructor"},{"inputs":[{"internalType":"address","name":"spender","type":"address"},{"internalType":"uint256","name":"allowance","type":"uint256"},{"internalType":"uint256","name":"needed","type":"uint256"}],"name":"ERC20InsufficientAllowance","type":"error"},{"inputs":[{"internalType":"address","name":"sender","type":"address"},{"internalType":"uint256","name":"balance","type":"uint256"},{"internalType":"uint256","name":"needed","type":"uint256"}],"name":"ERC20InsufficientBalance","type":"error"},{"inputs":[{"internalType":"address","name":"approver","type":"address"}],"name":"ERC20InvalidApprover","type":"error"},{"inputs":[{"internalType":"address","name":"receiver","type":"address"}],"name":"ERC20InvalidReceiver","type":"error"},{"inputs":[{"internalType":"address","name":"sender","type":"address"}],"name":"ERC20InvalidSender","type":"error"},{"inputs":[{"internalType":"address","name":"spender","type":"address"}],"name":"ERC20InvalidSpender","type":"error"},{"inputs":[{"internalType":"address","name":"owner","type":"address"}],"name":"OwnableInvalidOwner","type":"error"},{"inputs":[{"internalType":"address","name":"account","type":"address"}],"name":"OwnableUnauthorizedAccount","type":"error"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"owner","type":"address"},{"indexed":true,"internalType":"address","name":"spender","type":"address"},{"indexed":false,"internalType":"uint256","name":"value","type":"uint256"}],"name":"Approval","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"previousOwner","type":"address"},{"indexed":true,"internalType":"address","name":"newOwner","type":"address"}],"name":"OwnershipTransferred","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"from","type":"address"},{"indexed":true,"internalType":"address","name":"to","type":"address"},{"indexed":false,"internalType":"uint256","name":"value","type":"uint256"}],"name":"Transfer","type":"event"},{"inputs":[{"internalType":"address","name":"owner","type":"address"},{"internalType":"address","name":"spender","type":"address"}],"name":"allowance","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"spender","type":"address"},{"internalType":"uint256","name":"value","type":"uint256"}],"name":"approve","outputs":[{"internalType":"bool","name":"","type":"bool"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"address","name":"account","type":"address"}],"name":"balanceOf","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"decimals","outputs":[{"internalType":"uint8","name":"","type":"uint8"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"to","type":"address"},{"internalType":"uint256","name":"amount","type":"uint256"}],"name":"mint","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[],"name":"name","outputs":[{"internalType":"string","name":"","type":"string"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"owner","outputs":[{"internalType":"address","name":"","type":"address"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"renounceOwnership","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[],"name":"symbol","outputs":[{"internalType":"string","name":"","type":"string"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"totalSupply","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"to","type":"address"},{"internalType":"uint256","name":"value","type":"uint256"}],"name":"transfer","outputs":[{"internalType":"bool","name":"","type":"bool"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"address","name":"from","type":"address"},{"internalType":"address","name":"to","type":"address"},{"internalType":"uint256","name":"value","type":"uint256"}],"name":"transferFrom","outputs":[{"internalType":"bool","name":"","type":"bool"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"address","name":"newOwner","type":"address"}],"name":"transferOwnership","outputs":[],"stateMutability":"nonpayable","type":"function"}]'

abi_USTETH = '[{"inputs":[],"stateMutability":"nonpayable","type":"constructor"},{"inputs":[{"internalType":"address","name":"spender","type":"address"},{"internalType":"uint256","name":"allowance","type":"uint256"},{"internalType":"uint256","name":"needed","type":"uint256"}],"name":"ERC20InsufficientAllowance","type":"error"},{"inputs":[{"internalType":"address","name":"sender","type":"address"},{"internalType":"uint256","name":"balance","type":"uint256"},{"internalType":"uint256","name":"needed","type":"uint256"}],"name":"ERC20InsufficientBalance","type":"error"},{"inputs":[{"internalType":"address","name":"approver","type":"address"}],"name":"ERC20InvalidApprover","type":"error"},{"inputs":[{"internalType":"address","name":"receiver","type":"address"}],"name":"ERC20InvalidReceiver","type":"error"},{"inputs":[{"internalType":"address","name":"sender","type":"address"}],"name":"ERC20InvalidSender","type":"error"},{"inputs":[{"internalType":"address","name":"spender","type":"address"}],"name":"ERC20InvalidSpender","type":"error"},{"inputs":[{"internalType":"address","name":"owner","type":"address"}],"name":"OwnableInvalidOwner","type":"error"},{"inputs":[{"internalType":"address","name":"account","type":"address"}],"name":"OwnableUnauthorizedAccount","type":"error"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"owner","type":"address"},{"indexed":true,"internalType":"address","name":"spender","type":"address"},{"indexed":false,"internalType":"uint256","name":"value","type":"uint256"}],"name":"Approval","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"previousOwner","type":"address"},{"indexed":true,"internalType":"address","name":"newOwner","type":"address"}],"name":"OwnershipTransferred","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"from","type":"address"},{"indexed":true,"internalType":"address","name":"to","type":"address"},{"indexed":false,"internalType":"uint256","name":"value","type":"uint256"}],"name":"Transfer","type":"event"},{"inputs":[{"internalType":"address","name":"owner","type":"address"},{"internalType":"address","name":"spender","type":"address"}],"name":"allowance","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"spender","type":"address"},{"internalType":"uint256","name":"value","type":"uint256"}],"name":"approve","outputs":[{"internalType":"bool","name":"","type":"bool"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"address","name":"account","type":"address"}],"name":"balanceOf","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"decimals","outputs":[{"internalType":"uint8","name":"","type":"uint8"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"to","type":"address"},{"internalType":"uint256","name":"amount","type":"uint256"}],"name":"mint","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[],"name":"name","outputs":[{"internalType":"string","name":"","type":"string"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"owner","outputs":[{"internalType":"address","name":"","type":"address"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"renounceOwnership","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[],"name":"symbol","outputs":[{"internalType":"string","name":"","type":"string"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"totalSupply","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"to","type":"address"},{"internalType":"uint256","name":"value","type":"uint256"}],"name":"transfer","outputs":[{"internalType":"bool","name":"","type":"bool"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"address","name":"from","type":"address"},{"internalType":"address","name":"to","type":"address"},{"internalType":"uint256","name":"value","type":"uint256"}],"name":"transferFrom","outputs":[{"internalType":"bool","name":"","type":"bool"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"address","name":"newOwner","type":"address"}],"name":"transferOwnership","outputs":[],"stateMutability":"nonpayable","type":"function"}]'

pool_abi = '[{"inputs":[],"stateMutability":"nonpayable","type":"constructor"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"owner","type":"address"},{"indexed":true,"internalType":"int24","name":"tickLower","type":"int24"},{"indexed":true,"internalType":"int24","name":"tickUpper","type":"int24"},{"indexed":false,"internalType":"uint128","name":"amount","type":"uint128"},{"indexed":false,"internalType":"uint256","name":"amount0","type":"uint256"},{"indexed":false,"internalType":"uint256","name":"amount1","type":"uint256"}],"name":"Burn","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"owner","type":"address"},{"indexed":false,"internalType":"address","name":"recipient","type":"address"},{"indexed":true,"internalType":"int24","name":"tickLower","type":"int24"},{"indexed":true,"internalType":"int24","name":"tickUpper","type":"int24"},{"indexed":false,"internalType":"uint128","name":"amount0","type":"uint128"},{"indexed":false,"internalType":"uint128","name":"amount1","type":"uint128"}],"name":"Collect","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"sender","type":"address"},{"indexed":true,"internalType":"address","name":"recipient","type":"address"},{"indexed":false,"internalType":"uint128","name":"amount0","type":"uint128"},{"indexed":false,"internalType":"uint128","name":"amount1","type":"uint128"}],"name":"CollectProtocol","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"sender","type":"address"},{"indexed":true,"internalType":"address","name":"recipient","type":"address"},{"indexed":false,"internalType":"uint256","name":"amount0","type":"uint256"},{"indexed":false,"internalType":"uint256","name":"amount1","type":"uint256"},{"indexed":false,"internalType":"uint256","name":"paid0","type":"uint256"},{"indexed":false,"internalType":"uint256","name":"paid1","type":"uint256"}],"name":"Flash","type":"event"},{"anonymous":false,"inputs":[{"indexed":false,"internalType":"uint16","name":"observationCardinalityNextOld","type":"uint16"},{"indexed":false,"internalType":"uint16","name":"observationCardinalityNextNew","type":"uint16"}],"name":"IncreaseObservationCardinalityNext","type":"event"},{"anonymous":false,"inputs":[{"indexed":false,"internalType":"uint160","name":"sqrtPriceX96","type":"uint160"},{"indexed":false,"internalType":"int24","name":"tick","type":"int24"}],"name":"Initialize","type":"event"},{"anonymous":false,"inputs":[{"indexed":false,"internalType":"address","name":"sender","type":"address"},{"indexed":true,"internalType":"address","name":"owner","type":"address"},{"indexed":true,"internalType":"int24","name":"tickLower","type":"int24"},{"indexed":true,"internalType":"int24","name":"tickUpper","type":"int24"},{"indexed":false,"internalType":"uint128","name":"amount","type":"uint128"},{"indexed":false,"internalType":"uint256","name":"amount0","type":"uint256"},{"indexed":false,"internalType":"uint256","name":"amount1","type":"uint256"}],"name":"Mint","type":"event"},{"anonymous":false,"inputs":[{"indexed":false,"internalType":"uint8","name":"feeProtocol0Old","type":"uint8"},{"indexed":false,"internalType":"uint8","name":"feeProtocol1Old","type":"uint8"},{"indexed":false,"internalType":"uint8","name":"feeProtocol0New","type":"uint8"},{"indexed":false,"internalType":"uint8","name":"feeProtocol1New","type":"uint8"}],"name":"SetFeeProtocol","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"sender","type":"address"},{"indexed":true,"internalType":"address","name":"recipient","type":"address"},{"indexed":false,"internalType":"int256","name":"amount0","type":"int256"},{"indexed":false,"internalType":"int256","name":"amount1","type":"int256"},{"indexed":false,"internalType":"uint160","name":"sqrtPriceX96","type":"uint160"},{"indexed":false,"internalType":"uint128","name":"liquidity","type":"uint128"},{"indexed":false,"internalType":"int24","name":"tick","type":"int24"}],"name":"Swap","type":"event"},{"inputs":[{"internalType":"int24","name":"tickLower","type":"int24"},{"internalType":"int24","name":"tickUpper","type":"int24"},{"internalType":"uint128","name":"amount","type":"uint128"}],"name":"burn","outputs":[{"internalType":"uint256","name":"amount0","type":"uint256"},{"internalType":"uint256","name":"amount1","type":"uint256"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"address","name":"recipient","type":"address"},{"internalType":"int24","name":"tickLower","type":"int24"},{"internalType":"int24","name":"tickUpper","type":"int24"},{"internalType":"uint128","name":"amount0Requested","type":"uint128"},{"internalType":"uint128","name":"amount1Requested","type":"uint128"}],"name":"collect","outputs":[{"internalType":"uint128","name":"amount0","type":"uint128"},{"internalType":"uint128","name":"amount1","type":"uint128"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"address","name":"recipient","type":"address"},{"internalType":"uint128","name":"amount0Requested","type":"uint128"},{"internalType":"uint128","name":"amount1Requested","type":"uint128"}],"name":"collectProtocol","outputs":[{"internalType":"uint128","name":"amount0","type":"uint128"},{"internalType":"uint128","name":"amount1","type":"uint128"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[],"name":"factory","outputs":[{"internalType":"address","name":"","type":"address"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"fee","outputs":[{"internalType":"uint24","name":"","type":"uint24"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"feeGrowthGlobal0X128","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"feeGrowthGlobal1X128","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"recipient","type":"address"},{"internalType":"uint256","name":"amount0","type":"uint256"},{"internalType":"uint256","name":"amount1","type":"uint256"},{"internalType":"bytes","name":"data","type":"bytes"}],"name":"flash","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"uint16","name":"observationCardinalityNext","type":"uint16"}],"name":"increaseObservationCardinalityNext","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"uint160","name":"sqrtPriceX96","type":"uint160"}],"name":"initialize","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[],"name":"liquidity","outputs":[{"internalType":"uint128","name":"","type":"uint128"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"maxLiquidityPerTick","outputs":[{"internalType":"uint128","name":"","type":"uint128"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"recipient","type":"address"},{"internalType":"int24","name":"tickLower","type":"int24"},{"internalType":"int24","name":"tickUpper","type":"int24"},{"internalType":"uint128","name":"amount","type":"uint128"},{"internalType":"bytes","name":"data","type":"bytes"}],"name":"mint","outputs":[{"internalType":"uint256","name":"amount0","type":"uint256"},{"internalType":"uint256","name":"amount1","type":"uint256"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"uint256","name":"","type":"uint256"}],"name":"observations","outputs":[{"internalType":"uint32","name":"blockTimestamp","type":"uint32"},{"internalType":"int56","name":"tickCumulative","type":"int56"},{"internalType":"uint160","name":"secondsPerLiquidityCumulativeX128","type":"uint160"},{"internalType":"bool","name":"initialized","type":"bool"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"uint32[]","name":"secondsAgos","type":"uint32[]"}],"name":"observe","outputs":[{"internalType":"int56[]","name":"tickCumulatives","type":"int56[]"},{"internalType":"uint160[]","name":"secondsPerLiquidityCumulativeX128s","type":"uint160[]"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"bytes32","name":"","type":"bytes32"}],"name":"positions","outputs":[{"internalType":"uint128","name":"liquidity","type":"uint128"},{"internalType":"uint256","name":"feeGrowthInside0LastX128","type":"uint256"},{"internalType":"uint256","name":"feeGrowthInside1LastX128","type":"uint256"},{"internalType":"uint128","name":"tokensOwed0","type":"uint128"},{"internalType":"uint128","name":"tokensOwed1","type":"uint128"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"protocolFees","outputs":[{"internalType":"uint128","name":"token0","type":"uint128"},{"internalType":"uint128","name":"token1","type":"uint128"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"uint8","name":"feeProtocol0","type":"uint8"},{"internalType":"uint8","name":"feeProtocol1","type":"uint8"}],"name":"setFeeProtocol","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[],"name":"slot0","outputs":[{"internalType":"uint160","name":"sqrtPriceX96","type":"uint160"},{"internalType":"int24","name":"tick","type":"int24"},{"internalType":"uint16","name":"observationIndex","type":"uint16"},{"internalType":"uint16","name":"observationCardinality","type":"uint16"},{"internalType":"uint16","name":"observationCardinalityNext","type":"uint16"},{"internalType":"uint8","name":"feeProtocol","type":"uint8"},{"internalType":"bool","name":"unlocked","type":"bool"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"int24","name":"tickLower","type":"int24"},{"internalType":"int24","name":"tickUpper","type":"int24"}],"name":"snapshotCumulativesInside","outputs":[{"internalType":"int56","name":"tickCumulativeInside","type":"int56"},{"internalType":"uint160","name":"secondsPerLiquidityInsideX128","type":"uint160"},{"internalType":"uint32","name":"secondsInside","type":"uint32"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"recipient","type":"address"},{"internalType":"bool","name":"zeroForOne","type":"bool"},{"internalType":"int256","name":"amountSpecified","type":"int256"},{"internalType":"uint160","name":"sqrtPriceLimitX96","type":"uint160"},{"internalType":"bytes","name":"data","type":"bytes"}],"name":"swap","outputs":[{"internalType":"int256","name":"amount0","type":"int256"},{"internalType":"int256","name":"amount1","type":"int256"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"int16","name":"","type":"int16"}],"name":"tickBitmap","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"tickSpacing","outputs":[{"internalType":"int24","name":"","type":"int24"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"int24","name":"","type":"int24"}],"name":"ticks","outputs":[{"internalType":"uint128","name":"liquidityGross","type":"uint128"},{"internalType":"int128","name":"liquidityNet","type":"int128"},{"internalType":"uint256","name":"feeGrowthOutside0X128","type":"uint256"},{"internalType":"uint256","name":"feeGrowthOutside1X128","type":"uint256"},{"internalType":"int56","name":"tickCumulativeOutside","type":"int56"},{"internalType":"uint160","name":"secondsPerLiquidityOutsideX128","type":"uint160"},{"internalType":"uint32","name":"secondsOutside","type":"uint32"},{"internalType":"bool","name":"initialized","type":"bool"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"token0","outputs":[{"internalType":"address","name":"","type":"address"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"token1","outputs":[{"internalType":"address","name":"","type":"address"}],"stateMutability":"view","type":"function"}]'

abi_pool = '[{"inputs":[],"stateMutability":"nonpayable","type":"constructor"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"owner","type":"address"},{"indexed":true,"internalType":"int24","name":"tickLower","type":"int24"},{"indexed":true,"internalType":"int24","name":"tickUpper","type":"int24"},{"indexed":false,"internalType":"uint128","name":"amount","type":"uint128"},{"indexed":false,"internalType":"uint256","name":"amount0","type":"uint256"},{"indexed":false,"internalType":"uint256","name":"amount1","type":"uint256"}],"name":"Burn","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"owner","type":"address"},{"indexed":false,"internalType":"address","name":"recipient","type":"address"},{"indexed":true,"internalType":"int24","name":"tickLower","type":"int24"},{"indexed":true,"internalType":"int24","name":"tickUpper","type":"int24"},{"indexed":false,"internalType":"uint128","name":"amount0","type":"uint128"},{"indexed":false,"internalType":"uint128","name":"amount1","type":"uint128"}],"name":"Collect","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"sender","type":"address"},{"indexed":true,"internalType":"address","name":"recipient","type":"address"},{"indexed":false,"internalType":"uint128","name":"amount0","type":"uint128"},{"indexed":false,"internalType":"uint128","name":"amount1","type":"uint128"}],"name":"CollectProtocol","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"sender","type":"address"},{"indexed":true,"internalType":"address","name":"recipient","type":"address"},{"indexed":false,"internalType":"uint256","name":"amount0","type":"uint256"},{"indexed":false,"internalType":"uint256","name":"amount1","type":"uint256"},{"indexed":false,"internalType":"uint256","name":"paid0","type":"uint256"},{"indexed":false,"internalType":"uint256","name":"paid1","type":"uint256"}],"name":"Flash","type":"event"},{"anonymous":false,"inputs":[{"indexed":false,"internalType":"uint16","name":"observationCardinalityNextOld","type":"uint16"},{"indexed":false,"internalType":"uint16","name":"observationCardinalityNextNew","type":"uint16"}],"name":"IncreaseObservationCardinalityNext","type":"event"},{"anonymous":false,"inputs":[{"indexed":false,"internalType":"uint160","name":"sqrtPriceX96","type":"uint160"},{"indexed":false,"internalType":"int24","name":"tick","type":"int24"}],"name":"Initialize","type":"event"},{"anonymous":false,"inputs":[{"indexed":false,"internalType":"address","name":"sender","type":"address"},{"indexed":true,"internalType":"address","name":"owner","type":"address"},{"indexed":true,"internalType":"int24","name":"tickLower","type":"int24"},{"indexed":true,"internalType":"int24","name":"tickUpper","type":"int24"},{"indexed":false,"internalType":"uint128","name":"amount","type":"uint128"},{"indexed":false,"internalType":"uint256","name":"amount0","type":"uint256"},{"indexed":false,"internalType":"uint256","name":"amount1","type":"uint256"}],"name":"Mint","type":"event"},{"anonymous":false,"inputs":[{"indexed":false,"internalType":"uint8","name":"feeProtocol0Old","type":"uint8"},{"indexed":false,"internalType":"uint8","name":"feeProtocol1Old","type":"uint8"},{"indexed":false,"internalType":"uint8","name":"feeProtocol0New","type":"uint8"},{"indexed":false,"internalType":"uint8","name":"feeProtocol1New","type":"uint8"}],"name":"SetFeeProtocol","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"sender","type":"address"},{"indexed":true,"internalType":"address","name":"recipient","type":"address"},{"indexed":false,"internalType":"int256","name":"amount0","type":"int256"},{"indexed":false,"internalType":"int256","name":"amount1","type":"int256"},{"indexed":false,"internalType":"uint160","name":"sqrtPriceX96","type":"uint160"},{"indexed":false,"internalType":"uint128","name":"liquidity","type":"uint128"},{"indexed":false,"internalType":"int24","name":"tick","type":"int24"}],"name":"Swap","type":"event"},{"inputs":[{"internalType":"int24","name":"tickLower","type":"int24"},{"internalType":"int24","name":"tickUpper","type":"int24"},{"internalType":"uint128","name":"amount","type":"uint128"}],"name":"burn","outputs":[{"internalType":"uint256","name":"amount0","type":"uint256"},{"internalType":"uint256","name":"amount1","type":"uint256"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"address","name":"recipient","type":"address"},{"internalType":"int24","name":"tickLower","type":"int24"},{"internalType":"int24","name":"tickUpper","type":"int24"},{"internalType":"uint128","name":"amount0Requested","type":"uint128"},{"internalType":"uint128","name":"amount1Requested","type":"uint128"}],"name":"collect","outputs":[{"internalType":"uint128","name":"amount0","type":"uint128"},{"internalType":"uint128","name":"amount1","type":"uint128"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"address","name":"recipient","type":"address"},{"internalType":"uint128","name":"amount0Requested","type":"uint128"},{"internalType":"uint128","name":"amount1Requested","type":"uint128"}],"name":"collectProtocol","outputs":[{"internalType":"uint128","name":"amount0","type":"uint128"},{"internalType":"uint128","name":"amount1","type":"uint128"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[],"name":"factory","outputs":[{"internalType":"address","name":"","type":"address"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"fee","outputs":[{"internalType":"uint24","name":"","type":"uint24"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"feeGrowthGlobal0X128","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"feeGrowthGlobal1X128","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"recipient","type":"address"},{"internalType":"uint256","name":"amount0","type":"uint256"},{"internalType":"uint256","name":"amount1","type":"uint256"},{"internalType":"bytes","name":"data","type":"bytes"}],"name":"flash","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"uint16","name":"observationCardinalityNext","type":"uint16"}],"name":"increaseObservationCardinalityNext","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"uint160","name":"sqrtPriceX96","type":"uint160"}],"name":"initialize","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[],"name":"liquidity","outputs":[{"internalType":"uint128","name":"","type":"uint128"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"maxLiquidityPerTick","outputs":[{"internalType":"uint128","name":"","type":"uint128"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"recipient","type":"address"},{"internalType":"int24","name":"tickLower","type":"int24"},{"internalType":"int24","name":"tickUpper","type":"int24"},{"internalType":"uint128","name":"amount","type":"uint128"},{"internalType":"bytes","name":"data","type":"bytes"}],"name":"mint","outputs":[{"internalType":"uint256","name":"amount0","type":"uint256"},{"internalType":"uint256","name":"amount1","type":"uint256"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"uint256","name":"","type":"uint256"}],"name":"observations","outputs":[{"internalType":"uint32","name":"blockTimestamp","type":"uint32"},{"internalType":"int56","name":"tickCumulative","type":"int56"},{"internalType":"uint160","name":"secondsPerLiquidityCumulativeX128","type":"uint160"},{"internalType":"bool","name":"initialized","type":"bool"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"uint32[]","name":"secondsAgos","type":"uint32[]"}],"name":"observe","outputs":[{"internalType":"int56[]","name":"tickCumulatives","type":"int56[]"},{"internalType":"uint160[]","name":"secondsPerLiquidityCumulativeX128s","type":"uint160[]"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"bytes32","name":"","type":"bytes32"}],"name":"positions","outputs":[{"internalType":"uint128","name":"liquidity","type":"uint128"},{"internalType":"uint256","name":"feeGrowthInside0LastX128","type":"uint256"},{"internalType":"uint256","name":"feeGrowthInside1LastX128","type":"uint256"},{"internalType":"uint128","name":"tokensOwed0","type":"uint128"},{"internalType":"uint128","name":"tokensOwed1","type":"uint128"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"protocolFees","outputs":[{"internalType":"uint128","name":"token0","type":"uint128"},{"internalType":"uint128","name":"token1","type":"uint128"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"uint8","name":"feeProtocol0","type":"uint8"},{"internalType":"uint8","name":"feeProtocol1","type":"uint8"}],"name":"setFeeProtocol","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[],"name":"slot0","outputs":[{"internalType":"uint160","name":"sqrtPriceX96","type":"uint160"},{"internalType":"int24","name":"tick","type":"int24"},{"internalType":"uint16","name":"observationIndex","type":"uint16"},{"internalType":"uint16","name":"observationCardinality","type":"uint16"},{"internalType":"uint16","name":"observationCardinalityNext","type":"uint16"},{"internalType":"uint8","name":"feeProtocol","type":"uint8"},{"internalType":"bool","name":"unlocked","type":"bool"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"int24","name":"tickLower","type":"int24"},{"internalType":"int24","name":"tickUpper","type":"int24"}],"name":"snapshotCumulativesInside","outputs":[{"internalType":"int56","name":"tickCumulativeInside","type":"int56"},{"internalType":"uint160","name":"secondsPerLiquidityInsideX128","type":"uint160"},{"internalType":"uint32","name":"secondsInside","type":"uint32"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"recipient","type":"address"},{"internalType":"bool","name":"zeroForOne","type":"bool"},{"internalType":"int256","name":"amountSpecified","type":"int256"},{"internalType":"uint160","name":"sqrtPriceLimitX96","type":"uint160"},{"internalType":"bytes","name":"data","type":"bytes"}],"name":"swap","outputs":[{"internalType":"int256","name":"amount0","type":"int256"},{"internalType":"int256","name":"amount1","type":"int256"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"int16","name":"","type":"int16"}],"name":"tickBitmap","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"tickSpacing","outputs":[{"internalType":"int24","name":"","type":"int24"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"int24","name":"","type":"int24"}],"name":"ticks","outputs":[{"internalType":"uint128","name":"liquidityGross","type":"uint128"},{"internalType":"int128","name":"liquidityNet","type":"int128"},{"internalType":"uint256","name":"feeGrowthOutside0X128","type":"uint256"},{"internalType":"uint256","name":"feeGrowthOutside1X128","type":"uint256"},{"internalType":"int56","name":"tickCumulativeOutside","type":"int56"},{"internalType":"uint160","name":"secondsPerLiquidityOutsideX128","type":"uint160"},{"internalType":"uint32","name":"secondsOutside","type":"uint32"},{"internalType":"bool","name":"initialized","type":"bool"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"token0","outputs":[{"internalType":"address","name":"","type":"address"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"token1","outputs":[{"internalType":"address","name":"","type":"address"}],"stateMutability":"view","type":"function"}]'

def get_pool_address(tokenA, tokenB, fee,factory_contract):
    # Ensure tokens are in correct order (Uniswap V3 requires sorted token addresses)
    if tokenA > tokenB:
        tokenA, tokenB = tokenB, tokenA

    # Call the getPool function
    pool_address = factory_contract.functions.getPool(tokenA, tokenB, fee).call()
    return pool_address

USTETH_USTUSD_pool_address_1 = get_pool_address(USTUSD_address, USTETH_address, 100, factory_contract) #fee 0.01% = 100, fee 1% = 10000
USTETH_USTUSD_pool_address_2 = get_pool_address(USTUSD_address, USTETH_address, 10000, factory_contract)


print("Pool with fee 0.01% :",USTETH_USTUSD_pool_address_1)
print("Pool with fee 1% :",USTETH_USTUSD_pool_address_2)

In [ ]:
univ3_pool_1 = w3.eth.contract(address=USTETH_USTUSD_pool_address_1, abi=pool_abi)

current_price_1 = univ3_pool_1.functions.slot0().call()[0]**2 / 2**192

print(f"Price of USTETH in pool with fee 0.01% : {current_price_1} USTUSD")

univ3_pool_2 = w3.eth.contract(address=USTETH_USTUSD_pool_address_2, abi=pool_abi)

current_price_2 = univ3_pool_2.functions.slot0().call()[0]**2 / 2**192

print(f"Price of USTETH in pool with fee 1% : {current_price_2} USTUSD")


diff = abs(current_price_1 - current_price_2) / ((current_price_1 + current_price_2) / 2) * 100

print(f"Price difference between the two pools: {diff:.6f}%")


Given the prices above and that you start with some USTETH, how should you perform the swap?
Obviously, you should buy USTETH on the pool with the lower price and sell it on the pool with the higher price.

## Create a contract

To perform an atomic transaction, recall from week3 (Flight and hotel booking) that you need to deploy a contract with a function that performs multiple transactions and then call that function in your transaction. For this lab, you need to create and deploy a solidity contract on remix which interacts with the Uniswap V3 Router. By calling Uniswap's contracts with another contract on-chain, the transaction will not be between your original address but between the contract address. This means that, for your contract to work, it will provide the USTUSD tokens, so remember to approve your contract to use some USTUSD from your address. Additionally, the contract will automatically approve Uniswap upon creation.

We provide an **incomplete** solidity contract that does this arbitrage. Please fill in the contract and ask for help if it needs clarification. Deploy this contract onto the blockchain and try it yourself.

```solidity
// SPDX-License-Identifier: Unlicense

pragma solidity >=0.7.5;
pragma abicoder v2;

import "@uniswap/v3-periphery/contracts/interfaces/ISwapRouter.sol";
import "@uniswap/lib/contracts/libraries/TransferHelper.sol";
// Import the interface for the SwapRouter contract which handles V3 swaps
import "@uniswap/swap-router-contracts/contracts/interfaces/IV3SwapRouter.sol";

contract Arbitrage {

    // Maximum uint256 value used for infinite approval
    uint256 constant MAX_UINT = 2**256 - 1;

    // Address of the Uniswap V3 SwapRouter contract
    // This contract executes the actual token swaps
    address private constant SWAP_ROUTER = 0x3bFA4769FB09eefC5a80d6E87c3B9C650f7Ae48E;

    // Token Addresses
    // USTUSD: Likely a stablecoin pair or wrapped UST representation
    address private constant USTUSD = 0xc83B0efA5B3F13851DfA11de72EF6AFeF026730c;
    // USTETH: Likely a token representing UST-ETH LP or a specific derivative
    address public constant USTETH = 0x4E22e9951770b98d4f3B2BA6647f0f40A15A4057;

    // Immutable instance of the SwapRouter interface
    IV3SwapRouter public immutable swapRouter = IV3SwapRouter(SWAP_ROUTER);

    // Event to emit the final output amount after the arbitrage cycle
    event ArbitrageResult(uint256 amountOut);
    
    /**
     * @notice Executes a triangular arbitrage loop: USTUSD -> USTETH -> USTUSD
     * @param amountIn The amount of USTUSD tokens to start the arbitrage with
     * @return amountOut The final amount of USTUSD received after the full cycle
     */
    function doArbitrage(uint256 amountIn)
        external
        returns (uint256 amountOut)
    {
        // Step 1: Transfer the input tokens from the caller to this contract
        TransferHelper.safeTransferFrom(
            USTUSD,
            msg.sender,
            address(this),
            amountIn
        );

        // Step 2: Approve the SwapRouter to spend the input tokens (USTUSD)
        // This is required before calling exactInputSingle
        TransferHelper.safeApprove(USTUSD, address(swapRouter), amountIn);

        // --- LEG 1: Swap USTUSD -> USTETH ---
        
        // Define parameters for the first swap
        // fee: 100 corresponds to 0.01% (best for very stable pairs like stablecoins)
        // If the pair is volatile, a higher fee tier (e.g., 3000 for 0.3% or 10000 for 1%) might be needed.
        // You must ensure a pool with this specific fee tier exists for this token pair.
        IV3SwapRouter.ExactInputSingleParams memory params = IV3SwapRouter
            .ExactInputSingleParams({
                tokenIn: USTUSD,      // Selling USTUSD
                tokenOut: USTETH,     // Buying USTETH
                fee: ?,             // choose 0.01% fee tier or 1% fee tier? 
                recipient: address(this), // Receive intermediate tokens in this contract
                amountIn: amountIn,
                amountOutMinimum: 0,  // No slippage protection here (risky in production, usually set > 0)
                sqrtPriceLimitX96: 0  // No price limit constraint
            });

        // Execute the first swap
        uint256 amountOut1 = swapRouter.exactInputSingle(params);

        // Step 3: Approve the SwapRouter to spend the intermediate tokens (USTETH)
        // Using MAX_UINT for infinite approval to save gas on future transactions if reused
        TransferHelper.safeApprove(USTETH, address(swapRouter), MAX_UINT);

        // --- LEG 2: Swap USTETH -> USTUSD (Completing the loop) ---

        // Define parameters for the second swap
        // fee: 10000 corresponds to 1% (typically for very volatile or exotic pairs)
        // Ensure this matches the actual fee tier of the USTETH -> USTUSD pool on-chain.
        IV3SwapRouter.ExactInputSingleParams memory params2 = IV3SwapRouter
            .ExactInputSingleParams({
                tokenIn: USTETH,      // Selling USTETH
                tokenOut: USTUSD,     // Buying back USTUSD
                fee: ?,           // choose 0.01% fee tier or 1% fee tier? 
                recipient: msg.sender, // Send final profits directly back to the user
                amountIn: amountOut1, // Use all tokens received from the first swap
                amountOutMinimum: 0,  // No slippage protection (should be > amountIn to guarantee profit)
                sqrtPriceLimitX96: 0
            });

        // Execute the second swap
        amountOut = swapRouter.exactInputSingle(params2);

        // Emit the result for off-chain monitoring
        emit ArbitrageResult(amountOut);
        
        // Return the final amount
        return amountOut;
    }
}
```

## Instructions for deployment:
- Create a new project in [Remix](https://remix.ethereum.org/#lang=en&optimize&runs=200&evmVersion&version=soljson-v0.8.20+commit.a1b79de6.js)
- Compile the code
- Click on the deploy tab and select an Injected provider - Metamask on Sepolia
- Click on deploy - observe the transaction on Etherscan and fetch the contract address
- Verify the contract on Etherscan - Follow the steps from the assignments

## Approve USTETH and USTUSD to the arbitrage contract we just deployed

In [ ]:
# check allowance first before calling swap function
arbitrage_contract_address = ""

USTETH_contract = w3.eth.contract(address=USTETH_address, abi=abi_USTETH)
allowance_USTETH = USTETH_contract.functions.allowance(public_address, arbitrage_contract_address).call()

USTUSD_contract = w3.eth.contract(address=USTUSD_address, abi=abi_USTUSD)
allowance_USTUSD = USTUSD_contract.functions.allowance(public_address, arbitrage_contract_address).call()

print(f"Allowance of USTETH for arbitrage contract: {allowance_USTETH}")
print(f"Allowance of USTUSD for arbitrage contract: {allowance_USTUSD}")

In [ ]:
# approve arbitrage contract to USTETH with max allowance
allowance = 2**256 - 1  # max allowance
contract_function = USTETH_contract.functions.approve(
        arbitrage_contract_address,
        allowance
)

trx_params = contract_function.build_transaction(
        {
            "from": public_address,
            "gas": 500_000,
            "maxPriorityFeePerGas": w3.eth.max_priority_fee,
            "maxFeePerGas": 100 * 10**9,
            "type": '0x2',
            "chainId": 11155111,
            "value": 0,
            "nonce": w3.eth.get_transaction_count(public_address),
        }
    )

signed_txn = w3.eth.account.sign_transaction(trx_params, private_key)
raw_transaction = signed_txn.rawTransaction if hasattr(signed_txn, 'rawTransaction') else signed_txn.raw_transaction

trx_hash = w3.eth.send_raw_transaction(raw_transaction)
print(f"Arbitrage USTETH approve trx hash: {trx_hash.hex()}")

In [ ]:
# check allowance
allowance_after = USTETH_contract.functions.allowance(public_address, arbitrage_contract_address).call()
print(f"Allowance after approval: {allowance_after}")

In [ ]:
# approve arbitrage contract to USTUSD with max allowance
allowance = 2**256 - 1  # max allowance
contract_function = USTUSD_contract.functions.approve(
        arbitrage_contract_address,
        allowance
)

trx_params = contract_function.build_transaction(
        {
            "from": public_address,
            "gas": 500_000,
            "maxPriorityFeePerGas": w3.eth.max_priority_fee,
            "maxFeePerGas": 100 * 10**9,
            "type": '0x2',
            "chainId": 11155111,
            "value": 0,
            "nonce": w3.eth.get_transaction_count(public_address),
        }
    )

signed_txn = w3.eth.account.sign_transaction(trx_params, private_key)
raw_transaction = signed_txn.rawTransaction if hasattr(signed_txn, 'rawTransaction') else signed_txn.raw_transaction

trx_hash = w3.eth.send_raw_transaction(raw_transaction)
print(f"Arbitrage USTUSD approve trx hash: {trx_hash.hex()}")

In [ ]:
# check allowance
allowance_after = USTUSD_contract.functions.allowance(public_address, arbitrage_contract_address).call()
print(f"Allowance after approval: {allowance_after}")

## Call arbitrage function
- Once verified, click on ``Contract``->``Write Contract`` and call the ``doArbitrage`` function with ``amountIn`` as 1000 USTUSD (Type: 1000000000000000000000).
- Or you can directly call the function from Remix or Web3.py
- Check the transaction on Etherscan and observe the USTUSD received at the end of the transaction